# 车道定性对比（双模型 + GT + 类别）

在 **配置** 单元格里改 checkpoint、场景、segment 等；自上而下运行即可。

首格会用 `REPO_ROOT` + `in_repo()` 把 `config/...`、`output/...` 等**相对仓库根**的路径变成绝对路径，避免 notebook 工作目录不在仓库根时出现 `FileNotFoundError`。

底层复用 `qualitative_failure_analysis_compare_models.py`（网格/叠加/中文标签）。

**命令行一键复现（与下方单元格等价）**：在仓库根执行  
`python tools/pr/_run_qualitative_lane_compare_notebook_flow.py`（约数分钟，需 GPU + 数据 + 权重）。

In [1]:
import os
import sys


def _find_repo_root(start: str) -> str:
    p = os.path.abspath(start)
    for _ in range(8):
        if os.path.isdir(os.path.join(p, "unlanedet")):
            return p
        parent = os.path.dirname(p)
        if parent == p:
            break
        p = parent
    raise RuntimeError("找不到含 unlanedet/ 的仓库根，请在仓库内打开 notebook 或 chdir 到 tools/pr")


REPO_ROOT = _find_repo_root(os.getcwd())


def in_repo(rel: str) -> str:
    """相对路径一律相对仓库根；避免 notebook 的 cwd 不是仓库根时找不到 config/权重。"""
    return rel if os.path.isabs(rel) else os.path.join(REPO_ROOT, rel)


if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

_pr = os.path.join(REPO_ROOT, "tools", "pr")
if _pr not in sys.path:
    sys.path.insert(0, _pr)

import cv2
import numpy as np
import torch
from IPython.display import display, Image as IPyImage

import qualitative_failure_analysis_compare_models as M
from unlanedet.config import LazyConfig, instantiate
from unlanedet.checkpoint import Checkpointer
from unlanedet.data.build import build_batch_data_loader

print("REPO_ROOT =", REPO_ROOT)

REPO_ROOT = /data1/lxy_log/workspace/ms/UnLanedet


In [2]:
# ========== 配置：只改这里 ==========
M.DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

M.CFG_PATH = "config/llanetv1/openlane1000/category/clrnet_linear_resnet34.py"
M.MODEL2_CFG_PATH = "config/llanetv1/openlane1000/resnet34_llanet_ablation_exp2.py"
M.MODEL1_CKPT_PATH = "output/llanetv1/openlane1000/category/clrnet_linear_resnet34/model_best.pth"
M.MODEL2_CKPT_PATH = "output/llanetv1/openlane1000/resnet34_llanet_ablation_exp2/model_best.pth"

M.SEQUENCE_NAMES = ["curve"]
M.SEGMENT_PATH_CONTAINS = "segment-10203656353524179475_7625_000_7645_000_with_camera_labels"
M.OUT_DIR = os.path.join("output", "analysis", "curve", M.SEGMENT_PATH_CONTAINS)
M.APPEND_SCENARIO_SUBDIR = False

# 绘图/裁切（可按需调）
M.CROP_TOP_PX = 90
M.LANE_LABEL_FONT_SIZE = 22
M.HEADER_FONT_SIZE = 22
M.IOU_THRESHOLD = 0.5
M.IOU_WIDTH = 30
M.SUBSET_TOTAL_BATCH_SIZE = 4
M.SUBSET_NUM_WORKERS = 2

# 网格拼图
NUM_GRID_COLS = 8
CELL_W, CELL_H = 480, 270
GRID_POOL_MAX_FRAMES = 500
GRID_OUT_NAME = "qualitative_grid_compare.png"
MODEL1_DISPLAY_ZH = "CLRNet-linear"
MODEL2_DISPLAY_ZH = "LLANet Exp2"

# 单帧预览：在 union_indices 里的列下标（0 .. len-1）
PREVIEW_LOCAL_IDX = 0

In [3]:
def pick_column_indices(sorted_sel, n):
    u = len(sorted_sel)
    if n <= 0 or u == 0:
        return []
    if u <= n:
        return list(sorted_sel)
    if n == 1:
        return [sorted_sel[u // 2]]
    return [sorted_sel[int(round(i * (u - 1) / (n - 1)))] for i in range(n)]


def resolve_out_dir():
    seq = M.SEQUENCE_NAMES[0]
    rel = os.path.join(M.OUT_DIR, seq) if M.APPEND_SCENARIO_SUBDIR else M.OUT_DIR
    return in_repo(rel)

In [4]:
# 加载 config，筛帧，构造 union_indices（均匀列采样）
cfg = LazyConfig.load(in_repo(M.CFG_PATH))
param_cfg = getattr(cfg, "param_config", None)
assert param_cfg is not None, "param_config 缺失"

data_root = getattr(param_cfg, "data_root", None)
if not data_root:
    test_d0 = instantiate(cfg.dataloader.test).dataset
    data_root = test_d0.data_root

test_dataset = instantiate(cfg.dataloader.test.dataset)
seq_name = M.SEQUENCE_NAMES[0]
if len(M.SEQUENCE_NAMES) > 1:
    print("[WARN] 仅使用 SEQUENCE_NAMES[0]=", seq_name)

sel = M._match_indices_for_sequence(test_dataset, seq_name, data_root=data_root)
sel = sorted(set(sel))
sel = M._filter_indices_by_path_substr(test_dataset, sel, M.SEGMENT_PATH_CONTAINS)
pool = sel
if len(pool) > GRID_POOL_MAX_FRAMES:
    pool = M._subsample_evenly(pool, GRID_POOL_MAX_FRAMES)

union_indices = pick_column_indices(pool, NUM_GRID_COLS)
print(f"columns={len(union_indices)} idx={union_indices}")

subset_dataset = torch.utils.data.Subset(test_dataset, union_indices)
subset_dataloader = build_batch_data_loader(
    dataset=subset_dataset,
    total_batch_size=M.SUBSET_TOTAL_BATCH_SIZE,
    num_workers=M.SUBSET_NUM_WORKERS,
    drop_last=False,
    shuffle=False,
)

FileNotFoundError: [Errno 2] No such file or directory: 'config/llanetv1/openlane1000/category/clrnet_linear_resnet34.py'

In [ ]:
# 双模型推理 + 逐帧统计
m1_name = os.path.splitext(os.path.basename(M.MODEL1_CKPT_PATH))[0]
m2_name = os.path.splitext(os.path.basename(M.MODEL2_CKPT_PATH))[0]

model1 = instantiate(cfg.model).to(M.DEVICE)
model1.eval()
Checkpointer(model1).load(in_repo(M.MODEL1_CKPT_PATH))
pred1 = M._run_inference_on_subset(model1, subset_dataloader, device=M.DEVICE)
del model1
torch.cuda.empty_cache() if torch.cuda.is_available() else None

cfg1_abs = in_repo(M.CFG_PATH)
cfg2_abs = in_repo(M.MODEL2_CFG_PATH or M.CFG_PATH)
cfg2 = LazyConfig.load(cfg2_abs) if cfg2_abs != cfg1_abs else cfg
model2 = instantiate(cfg2.model).to(M.DEVICE)
model2.eval()
Checkpointer(model2).load(in_repo(M.MODEL2_CKPT_PATH))
pred2 = M._run_inference_on_subset(model2, subset_dataloader, device=M.DEVICE)
del model2
torch.cuda.empty_cache() if torch.cuda.is_available() else None

assert len(pred1) == len(pred2) == len(union_indices)

pred_stats_m1, pred_stats_m2 = [], []
for local_idx, orig_idx in enumerate(union_indices):
    data_info = test_dataset.data_infos[orig_idx]
    gt_lanes = data_info.get("lanes", [])
    gt_cats = data_info.get("lane_categories", [-1] * len(gt_lanes))
    pred_stats_m1.append(
        M._compute_frame_match_stats(
            pred_lanes=pred1[local_idx],
            gt_lanes=gt_lanes,
            param_cfg=param_cfg,
            gt_cats=gt_cats,
            iou_threshold=M.IOU_THRESHOLD,
            iou_width=M.IOU_WIDTH,
        )
    )
    param_for_m2 = getattr(cfg2, "param_config", None) or param_cfg
    pred_stats_m2.append(
        M._compute_frame_match_stats(
            pred_lanes=pred2[local_idx],
            gt_lanes=gt_lanes,
            param_cfg=param_for_m2,
            gt_cats=gt_cats,
            iou_threshold=M.IOU_THRESHOLD,
            iou_width=M.IOU_WIDTH,
        )
    )
print("pred_stats ready, frames:", len(pred_stats_m1))

## 单帧：三模型叠加 + 类别（写盘并在 notebook 中预览）

修改上一格里的 `PREVIEW_LOCAL_IDX` 后只重跑本格即可。

In [ ]:
li = int(min(max(PREVIEW_LOCAL_IDX, 0), len(union_indices) - 1))
orig_idx = union_indices[li]
data_info = test_dataset.data_infos[orig_idx]
img_path = M._img_path_from_data_info(data_info)
img_bgr = cv2.imread(img_path)
assert img_bgr is not None, img_path

gt_s, s1, s2 = pred_stats_m1[li], pred_stats_m1[li], pred_stats_m2[li]
out_dir = resolve_out_dir()
os.makedirs(out_dir, exist_ok=True)
safe = (M._img_name_from_data_info(data_info) or "frame").replace("/", "_")
single_path = os.path.join(out_dir, f"{safe}__compare_notebook.png")

M._draw_failure_comparison(
    out_path=single_path,
    img_bgr=img_bgr,
    gt_stats=gt_s,
    m1_stats=s1,
    m2_stats=s2,
    m1_name=m1_name,
    m2_name=m2_name,
)
print("saved", single_path)
display(IPyImage(filename=single_path))

## 多列网格（论文式拼图）

行：原图 | 模型1 | 模型2 | GT。

In [ ]:
rows = [
    ("raw", "输入图像"),
    ("m1", f"{MODEL1_DISPLAY_ZH} 预测（红·含类别）"),
    ("m2", f"{MODEL2_DISPLAY_ZH} 预测（蓝·含类别）"),
    ("gt", "真值 GT（绿·车道类别）"),
]
row_blocks = []
for mode_key, row_title in rows:
    cells = []
    for li, orig_idx in enumerate(union_indices):
        data_info = test_dataset.data_infos[orig_idx]
        ip = M._img_path_from_data_info(data_info)
        im = cv2.imread(ip)
        cells.append(
            M.render_grid_cell_panel(
                im,
                mode_key,
                pred_stats_m1[li],
                pred_stats_m1[li],
                pred_stats_m2[li],
                m1_display=MODEL1_DISPLAY_ZH,
                m2_display=MODEL2_DISPLAY_ZH,
            )
        )
    row_blocks.append(M.compose_grid_row(row_title, cells, CELL_W, CELL_H))

grid_bgr = cv2.vconcat(row_blocks)
out_dir = resolve_out_dir()
os.makedirs(out_dir, exist_ok=True)
grid_path = os.path.join(out_dir, GRID_OUT_NAME)
cv2.imwrite(grid_path, grid_bgr)
print("saved", grid_path)
display(IPyImage(filename=grid_path))

## 可选：按「失败帧」批量导出（原版 `M.main()`）

需要时取消注释并运行。会再跑一轮完整子集与多图输出，耗时较长。

In [ ]:
# M.MAX_FRAMES_PER_SEQUENCE = 120
# M.TOPK_FAILURES_TO_DRAW_PER_SEQUENCE = 12
# M.ALSO_DRAW_RANDOM = 6
# M.main()